# Model 2



Using $\text{Model}_{\text{pref}}$ as the base model, include one additional
alternative attribute and one interaction of a socioeconomic characteristic
with either the ASCs or one of the attributes. Present both the
specification and the estimation results (as defined previously).

# Importing packages

We first import some useful Python packages. pandas is an open source data analysis and manipulation tool built in the Python programming language. Biogeme is a Python package for estimating discrete choice models to explain and predict how people choose among alternatives.

In [1]:
import pandas as pd
import biogeme.database as db
import biogeme.biogeme as bio
from tqdm.notebook import tqdm
from biogeme.expressions import Beta, Variable, log, exp
from biogeme import models
from biogeme.results_processing import EstimationResults
from biogeme.results_processing import get_pandas_estimated_parameters, html_output

g++ not available, if using conda: `conda install gxx`
c:\Users\chloe\Desktop\3A\Data Science\Modelling Choice Behavior\Biogeme\myvenv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# Reading and preparing the data

We first import the data into Pandas. We read the data from a text file, where the data are separated by tabs. 

In [2]:
df = pd.read_csv('lpmc19.dat', sep='\t')
df

,trip_id,household_id,person_n,trip_n,travel_mode,purpose,fueltype,faretype,bus_scale,survey_year,...,dur_pt_access,dur_pt_rail,dur_pt_bus,dur_pt_int,pt_interchanges,dur_driving,cost_transit,cost_driving_fuel,cost_driving_ccharge,driving_traffic_percent
0,5,0,1,3,4,3,1,4,1.0,1,...,0.205556,0.000000,0.221111,0.000000,0,0.222222,1.5,0.77,0.0,0.097500
1,16,2,1,2,4,3,1,1,1.0,1,...,0.098611,0.000000,0.058333,0.000000,0,0.058611,1.5,0.16,0.0,0.123223
2,26,7,0,2,1,3,2,1,1.0,1,...,0.068333,0.000000,0.033889,0.000000,0,0.055278,1.5,0.10,0.0,0.105528
3,27,7,1,0,4,4,2,1,1.0,1,...,0.193889,0.516667,0.000000,0.100000,1,0.340833,1.5,2.04,0.0,0.280359
4,52,12,1,2,4,5,2,1,1.0,1,...,0.062500,0.000000,0.491944,0.094722,1,0.355556,3.0,1.19,0.0,0.249219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,81040,17604,2,0,3,1,1,2,0.0,3,...,0.264444,0.000000,0.353333,0.176667,1,0.288889,0.0,0.85,0.0,0.175000
4996,81059,17606,1,2,3,3,6,1,1.0,3,...,0.100278,0.000000,0.360278,0.000000,0,0.281667,1.5,0.63,0.0,0.384615
4997,81062,17607,0,4,3,5,6,1,1.0,3,...,0.069444,0.000000,0.166389,0.000000,0,0.146111,1.5,0.28,0.0,0.653992
4998,81071,17610,0,0,4,3,2,1,1.0,3,...,0.206111,0.000000,0.400000,0.000000,0,0.259167,1.5,0.88,0.0,0.233655


We then import this database into Biogeme.

In [3]:
database = db.Database('netherlands', df)

We identify the columns that will be used as variables in our model. 

In [4]:
trip_id = Variable('trip_id')
travel_mode = Variable('travel_mode')
purpose = Variable('purpose')
distance = Variable('distance')
dur_walking = Variable('dur_walking')
dur_cycling = Variable('dur_cycling')
dur_pt_access = Variable('dur_pt_access')
dur_pt_rail  = Variable('dur_pt_rail')
dur_pt_bus  = Variable('dur_pt_bus')
dur_pt_int = Variable('dur_pt_int')
dur_driving = Variable('dur_driving')
cost_transit = Variable('cost_transit')
cost_driving_fuel = Variable('cost_driving_fuel')
cost_driving_ccharge = Variable('cost_driving_ccharge')
driving_traffic_percent = Variable('driving_traffic_percent')
age = Variable('age')

In [5]:
database.num_rows()

5000

We define the total travel time by public transportation

In [6]:
dur_pt_total = dur_pt_access + dur_pt_rail + dur_pt_bus + dur_pt_int
cost_car = cost_driving_fuel + cost_driving_ccharge

We convert the Guilders into Euros.

In [7]:
POUNDS_TO_EUROS = 1.16
cost_car_euro = cost_car * POUNDS_TO_EUROS
cost_pt_euro = cost_transit * POUNDS_TO_EUROS

In [8]:
df['age'].dropna()


0       54
1       41
2       49
3       51
4       56
        ..
4995    17
4996    33
4997    38
4998    33
4999    40
Name: age, Length: 5000, dtype: int64

# Model specification

We can now specify a model that contains 4 ASC

In [9]:
asc_car = Beta('asc_car', 0, None, None, 0)
asc_walk = Beta('asc_walk', 0, None, None, 0)
asc_bike = Beta('asc_bike', 0, None, None, 0)
asc_pt = Beta('asc_pt', 0, None, None, 0)
beta_cost = Beta('beta_cost_car', 0, None, None, 0)
beta_age = Beta('beta_age', 0, None, None, 0)
beta_time_car = Beta('beta_time_car', 0, None, None, 0)
beta_time_walk = Beta('beta_time_walk', 0, None, None, 0)
beta_time_bike = Beta('beta_time_bike', 0, None, None, 0)
beta_time_pt = Beta('beta_time_pt', 0, None, None, 0)
beta_driving_traffic_percent = Beta('beta_driving_traffic_percent', 0, None, None, 0)

We  write the utility functions:
**We choose walking as the reference**

In [10]:
v_car = asc_car + beta_age * age + beta_cost * cost_car_euro + beta_time_car * dur_driving + beta_driving_traffic_percent * driving_traffic_percent
v_walk = 0 + beta_time_walk * dur_walking
v_bike = asc_bike + beta_age * age + beta_time_bike * dur_cycling
v_pt = asc_pt + beta_age * age + beta_cost * cost_pt_euro + beta_time_pt * dur_pt_total

And we write the choice model:

In [11]:
V = {1: v_walk, 2: v_bike, 3: v_pt, 4: v_car}

Blabla

In [12]:
logprob = models.loglogit(V, None, travel_mode)

We initialize Biogeme with this expression and the database. We call the model 'binary_netherlands'.

In [13]:
biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = 'model_2'

# Model estimation

Biogeme tries to read a file `__binary_netherlands.iter` containing intermediary results from a previous estimation run. If it does not find it, it triggers a warning that can be ignored.

In [14]:
results_model_2 = biogeme.estimate()

We print some general statistics: 

In [15]:
print(results_model_2.print_general_statistics())

Number of estimated parameters             10
Sample size                                5000
Excluded observations                      0
Init log likelihood                        -6931.472
Final log likelihood                       -4127.624
Likelihood ratio test for the init. model  5607.696
Rho-square for the init. model             0.405
Rho-square-bar for the init. model         0.403
Akaike Information Criterion               8275.248
Bayesian Information Criterion             8340.42
Final gradient norm                        1.5331E-03
Bootstrapping time                         None


We print the estimation results

In [16]:
get_pandas_estimated_parameters(estimation_results=results_model_2)

{'Estimated parameters':                            Name     Value  Robust std err.  Robust t-stat.  \
 0                beta_time_walk -8.734141         0.354264      -24.654355   
 1                      asc_bike -5.290113         0.222223      -23.805406   
 2                      beta_age  0.009964         0.002196        4.537687   
 3                beta_time_bike -5.581297         0.527957      -10.571490   
 4                        asc_pt -3.037818         0.156417      -19.421283   
 5                 beta_cost_car -0.122944         0.016541       -7.432534   
 6                  beta_time_pt -3.269556         0.244009      -13.399342   
 7                       asc_car -2.025265         0.153844      -13.164379   
 8                 beta_time_car -4.828793         0.386810      -12.483643   
 9  beta_driving_traffic_percent -2.869884         0.236802      -12.119320   
 
    Robust p-value  
 0    0.000000e+00  
 1    0.000000e+00  
 2    5.687470e-06  
 3    0.000000e+00  


# Test the Null hypothesis that Model 1 is true model compared to Model 2.


In [17]:
results_model_1 = EstimationResults.from_yaml_file(filename='model_1.yaml')

general_statistics_model_1 = results_model_1.get_general_statistics()
general_statistics_model_1

general_statistics_model_2 = results_model_2.get_general_statistics()

LR = float(general_statistics_model_1['Final log likelihood'])
LU = float(general_statistics_model_2['Final log likelihood'])
test = -2 * (LR - LU)
test

KR = float(general_statistics_model_1['Number of estimated parameters'])
KU = float(general_statistics_model_2['Number of estimated parameters'])
degrees_of_freedom = KU - KR
degrees_of_freedom

from scipy.stats import chi2
threshold = chi2.ppf(.95, degrees_of_freedom)
threshold

results_model_2.likelihood_ratio_test(results_model_1, 0.05)

LRTuple(message='H0 can be rejected at level 5.0%', statistic=187.18292091446892, threshold=np.float64(5.991464547107979))